# golars in gopyter

This notebook's cells are [golars](https://github.com/Gaurav-Gosain/golars) scripts (`.glr`), a small pipeline language for DataFrames. They run in one `golars kernel-host` session, so frames loaded in one cell are there in the next. A cell that changes the focused frame shows it as a table; `show`, `head` and `tail` print tables too.

It needs `golars` on your `PATH` (or `GOLARS_BIN`). Paths are relative to the notebook's directory: `orders.csv` and `customers.csv` sit next to it. Run it all with `A`, or headless with `gopyter run examples/golars.ipynb`.

In [ ]:
load orders.csv

## Filter and derive

Pipeline commands add to the focused frame's lazy plan. Tall tables show their first and last rows, like polars.

In [ ]:
filter discount is_not_null and qty > 1
with revenue = unit_price * qty * (1 - discount)

## Group

`stash` keeps the cleaned frame under a name, so later cells can branch from it with `use`.

In [ ]:
stash clean
groupby region revenue:sum:total revenue:count:orders
sort total desc

## Join

`load ... as NAME` stages a second frame without moving the focus.

In [ ]:
load customers.csv as customers
use clean
join customers on customer
select order_id name tier product revenue
sort revenue desc
head 5

## Branch again

Each `use` starts from a copy of the stash.

In [ ]:
use clean
groupby product qty:sum:units
sort units desc
show

In [ ]:
use clean
describe

## Expressions

Expressions call golars functions by name, as methods or functions (`x.round(2)` or `round(x, 2)`, `str.to_uppercase(region)`), with lists, keyword arguments, `//`, `%`, `in [...]` and `when ... then ... otherwise`.

In [ ]:
use clean
with band = when revenue > 100 then "big" otherwise "small"
with size = cut(qty, [2, 5], labels=["few", "some", "many"])
with bucket = revenue // 50 * 50
with code = concat_str(str.to_uppercase(region), product.str.slice(0, 3), separator="-")
filter region in ["eu", "us"] and order_id % 2 == 1
select order_id, code, band, size, bucket